# Scenario 1: A single ML scientist competing in an ML competition

MLflow setup:
- Tracking server: no
- Backend store: ~~local filesystem~~ sqlite (the local filesystem backend has been deprecated since February 2026)
- Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow server.

```bash
# Assuming this is executed from the running-mlflow-examples folder
uvx mlflow==3.10.1 server --default-artifact-root "file:///$(pwd)/scenario-1-artifacts" --artifacts-destination "file:///$(pwd)/scenario-1-artifacts" --backend-store-uri "sqlite:///$(pwd)/scenario-1-mlflow.db"
```

In [1]:
import os
import mlflow

In [2]:
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

Tracking URI: sqlite:////Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlflow.db


By default is uses the mlflow.db sqlite database, so we need to change it

In [3]:
mlflow.set_tracking_uri(f"sqlite:///{os.getcwd()}/scenario-1-mlflow.db")
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

Tracking URI: sqlite:////Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/scenario-1-mlflow.db


In [4]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:////Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/scenario-1-artifacts/0', creation_time=1776013431681, experiment_id='0', last_update_time=1776013431681, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

Creating an experiment and logging a new run

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment('my-experiment-1')

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {'C': 0.1, 'random_state': 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric('accuracy', accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name='models')
    print(f'default artifacts URI: {mlflow.get_artifact_uri()}')

2026/04/12 14:04:18 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/04/12 14:04:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: /Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1/ba619dec8b494c0e92c008774d8457e7/artifacts


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='/Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1', creation_time=1776013458793, experiment_id='1', last_update_time=1776013458793, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location='file:////Users/danielwohlgemuth/Code/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/scenario-1-artifacts/0', creation_time=1776013431681, experiment_id='0', last_update_time=1776013431681, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

## Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

In [9]:
from mlflow.exceptions import MlflowException

client.search_registered_models()

[]